# Spectral Convergence: Legendre vs Chebyshev

Reads results from `outputs/spectral_convergence/` (generated by
`tests/scripts/zoomy_core/swe/run_spectral_convergence.py`).

In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DATA_DIR = "outputs/spectral_convergence"

## Load results

In [3]:
manifest_path = os.path.join(DATA_DIR, "outputs/spectral_convergence/manifest.json")
if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"Run the spectral convergence script first:\n"
                            f"  python tests/scripts/zoomy_core/swe/run_spectral_convergence.py")

with open(manifest_path) as f:
    manifest = json.load(f)

results = {}
for key in manifest:
    path = os.path.join(DATA_DIR, f"{key}.json")
    if os.path.exists(path):
        with open(path) as f:
            results[key] = json.load(f)

leg = {r["level"]: r for k, r in results.items() if "Legendre" in k and "error" not in r}
cheb = {r["level"]: r for k, r in results.items() if "Chebyshev" in k and "error" not in r}

print(f"Legendre levels: {sorted(leg.keys())}")
print(f"Chebyshev levels: {sorted(cheb.keys())}")

FileNotFoundError: Run the spectral convergence script first:
  python tests/scripts/zoomy_core/swe/run_spectral_convergence.py

## Timing table

In [ ]:
print(f"{'Basis':<12s} {'L':>2s} {'n_vars':>6s} {'build':>7s} {'solve':>7s} {'h_min':>8s} {'h_max':>8s} {'status':<10s}")
print("-" * 65)

for level in range(max(max(leg.keys(), default=0), max(cheb.keys(), default=0)) + 1):
    for store, bname in [(leg, "Legendre"), (cheb, "Chebyshev")]:
        if level in store:
            r = store[level]
            print(f"{bname:<12s} {level:>2d} {r['n_vars']:>6d} {r['build_time']:>6.1f}s "
                  f"{r['solve_time']:>6.1f}s {r['h_min']:>8.4f} {r['h_max']:>8.4f} {'OK':<10s}")
        else:
            key = f"{bname}_L{level}_{'orthogonal' if bname=='Legendre' else 'physical'}"
            m = manifest.get(key, {})
            err = m.get("error", m.get("ok", "missing"))
            if err:
                print(f"{bname:<12s} {level:>2d} {'—':>6s} {m.get('build_time',-1):>6.1f}s "
                      f"{'—':>7s} {'—':>8s} {'—':>8s} {str(err)[:20]}")

## Build and solve time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for store, label, marker in [(leg, "Legendre", "o"), (cheb, "Chebyshev", "s")]:
    levels = sorted(store.keys())
    axes[0].plot(levels, [store[l]["build_time"] for l in levels], f"{marker}-", label=label)
    axes[1].plot(levels, [store[l]["solve_time"] for l in levels], f"{marker}-", label=label)

axes[0].set_xlabel("Level"); axes[0].set_ylabel("Time (s)"); axes[0].set_title("Build time")
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_yscale("log")
axes[1].set_xlabel("Level"); axes[1].set_ylabel("Time (s)"); axes[1].set_title(f"Solve time (t=0.3, 30 cells)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.savefig(os.path.join(DATA_DIR, "timing.png"), dpi=150)
print("Saved: timing.png")

## Water depth convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
cmap = plt.cm.viridis

for ax, store, title in [(axes[0], leg, "Legendre"), (axes[1], cheb, "Chebyshev (physical weight)")]:
    levels = sorted(store.keys())
    for l in levels:
        r = store[l]
        color = cmap(l / max(max(levels, default=1), 1))
        ax.plot(r["x"], r["h"], color=color, linewidth=1.2, label=f"L{l}")
    ax.set_title(title); ax.set_xlabel("x"); ax.set_ylabel("h")
    ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

fig.suptitle("Water depth convergence with level", fontsize=13)
plt.savefig(os.path.join(DATA_DIR, "h_convergence.png"), dpi=150)
print("Saved: h_convergence.png")

## Velocity profiles: one row per level, columns = x positions

In [ ]:
all_levels = sorted(set(leg.keys()) | set(cheb.keys()))
n_levels = len(all_levels)

fig, axes = plt.subplots(n_levels, 2, figsize=(10, 2.5 * n_levels), constrained_layout=True, squeeze=False)

for row, level in enumerate(all_levels):
    for col, x_pos in enumerate(["-1.0", "1.0"]):
        ax = axes[row, col]
        for store, bname, color in [(leg, "Legendre", "#1f77b4"), (cheb, "Chebyshev", "#d62728")]:
            if level in store and "profiles" in store[level]:
                p = store[level]["profiles"].get(x_pos)
                if p:
                    ax.plot(p["u"], p["zeta"], color=color, linewidth=1.3, label=bname)
        ax.set_title(f"L{level} at x={x_pos}"); ax.set_xlabel("u"); ax.set_ylabel("zeta")
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

fig.suptitle("Velocity profiles: Legendre vs Chebyshev at each level", fontsize=13)
plt.savefig(os.path.join(DATA_DIR, "velocity_profiles.png"), dpi=150)
print("Saved: velocity_profiles.png")

## Profile convergence: selected levels overlaid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
selected = [0, 1, 2, 4, max(all_levels)] if all_levels else [0]
selected = sorted(set(l for l in selected if l in set(leg.keys()) | set(cheb.keys())))
cmap2 = plt.cm.plasma

for col, x_pos in enumerate(["-1.0", "1.0"]):
    ax = axes[col]
    for i, level in enumerate(selected):
        color = cmap2(i / max(len(selected) - 1, 1))
        for store, bname, ls in [(leg, "Leg", "-"), (cheb, "Cheb", "--")]:
            if level in store and "profiles" in store[level]:
                p = store[level]["profiles"].get(x_pos)
                if p:
                    ax.plot(p["u"], p["zeta"], color=color, linestyle=ls, linewidth=1.3,
                            label=f"{bname} L{level}")
    ax.set_title(f"x = {x_pos}"); ax.set_xlabel("u"); ax.set_ylabel("zeta")
    ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

fig.suptitle("Velocity profile convergence with level", fontsize=13)
plt.savefig(os.path.join(DATA_DIR, "profile_convergence.png"), dpi=150)
print("Saved: profile_convergence.png")